# 第2回 宿題（発展）: 用語集 Q&A を LoRA で学習し、学習前後を比較する

講座の用語集から作った Q&A データ（`glossary_sft.jsonl`、116 件）で小さな指示チューニング済みモデルを LoRA 学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。演習のノートブック（`lora_format_tuning.ipynb`）を書き換えたものです。

**演習との違い**

| | 演習 | この宿題 |
| --- | --- | --- |
| 学習データ | 架空 FAQ 5 項目（14 件） | 用語集 58 語 × 質問 2 種（116 件） |
| プロンプト | FAQ ＋ 質問（FAQ は入力で渡す） | 質問だけ（知識はモデルの重みだけで答える） |
| 望ましい出力 | `answer` / `needs_review` の JSON | 用語集の定義文（平文） |
| 評価 | 形式遵守率と内容正解率 | 質問の種類ごとの正解率（下の 3 種類） |

**評価用の質問（test）の 3 種類**

1. `same`: 学習に使った質問そのまま。学習データを再現できるかの確認。
2. `paraphrase`: 用語集にある語を、学習とは違う言い回しで聞く質問。言い換えに対応できるかの確認。
3. `unknown`: 用語集にない語を聞く質問。学習データには「分かりません」と答える例を**入れていない**ので、作り話をするか、断るかを観察する。

**進め方**: セルは上から順に実行します。使う前に、Colab の左側のファイル一覧に `glossary_sft.jsonl` をアップロードしてください（`/content/glossary_sft.jsonl`）。

**使うもの**: Google Colab（T4 GPU）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [2]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [3]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の `result.json` にそのまま記録されます。

- `num_train_epochs: 5` は、学習データを 5 周する設定です。演習の 20 周は 14 件の少ないデータ向けで、今回は約 110 件あるので、まず 5 周から始めます（1 周が約 14 ステップ、合計約 70 ステップ）。
- `max_new_tokens: 256`: 回答は定義文（最大 200 文字ほど）なので、演習の 96 より長く取っています。
- `glossary_sft_path`: アップロードした学習データの場所です。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃えます。

In [4]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,   # 学習データを何周学習するか。約 110 件、バッチ 4 × 蓄積 2 で 1 周約 14 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "glossary_sft_path": "/content/glossary_sft.jsonl",   # アップロードした学習データ
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "glossary_sft_path": "/content/glossary_sft.jsonl",
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 学習データと評価データ

**やること**: `glossary_sft.jsonl` を読み込み、学習用（train）と検証用（valid）に分けます。評価用の質問（test）とその正解（gold）は、このセルに直接書きます。

**データの形式**

- 学習データ（train / valid）: `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` の 1 行 1 件。`glossary_sft.jsonl` がすでにこの形なので、変換せずそのまま `Dataset` にします。
- 評価データ（test）: `{"question": 質問, "category": 種類}` のリスト。
- 正解（gold）: 質問をキーにした辞書。`same` と `paraphrase` は答えに含まれるべき語句（`must_include`）、`unknown` は空です。モデルには渡しません。

**データの分け方**

- valid: 全 116 件のうち 20 件ごとに 1 件（6 件）。学習中の loss を見るためだけに使い、学習には使いません。最終的な評価には使いません。
- train: 残りの 110 件。
- test: 学習に使っていない質問 `paraphrase`（15 問）と `unknown`（8 問）、学習に使った質問そのまま `same`（10 問）の計 33 問。`same` だけは train に含まれる質問です（暗記の確認が目的）。

**`unknown` の質問**: 用語集のどこにも出てこない語を選んでいます。セルの中で、その語が `glossary_sft.jsonl` に出てこないことを確かめます。`unknown` は正解の文が無いので、答えを「断った（declined）」か「答えた（answered）」かに分類し、内容は表で自分の目で確かめます。

**見るところ**: `train=110 valid=6 test=33` のように件数が表示されます。確認用のチェック（`assert`）に失敗した場合は、質問や語句の指定を見直してください。

In [5]:
import json
rows = [json.loads(l) for l in open(CONFIG["glossary_sft_path"], encoding="utf-8")]
valid_rows = rows[1::20]                                    # 20 件ごとに 1 件を検証用にする
train_rows = [r for i, r in enumerate(rows) if i % 20 != 1]

def q_of(r): return r["messages"][0]["content"]
def a_of(r): return r["messages"][1]["content"]
train_q = {q_of(r): a_of(r) for r in train_rows}
valid_q = {q_of(r) for r in valid_rows}
all_text = "".join(q_of(r) + a_of(r) for r in rows)         # 用語集に出てくる全文字列

# (質問, 答えに含まれるべき語句)
# same: 学習に使った質問そのまま（学習データを再現できるか）
SAME = [
    ("「トークン」とは何ですか？", ["最小単位"]),
    ("「過学習」とは何ですか？", ["学習データ", "検証データ"]),
    ("「ハルシネーション」とは何ですか？", ["もっともらしく"]),
    ("「APIキー」とは何ですか？", ["秘密"]),
    ("「量子化」とは何ですか？", ["ビット数", "メモリ"]),
    ("「エポック」とは何ですか？", ["1周", "過学習"]),
    ("「ガードレール」とは何ですか？", ["制約", "検査"]),
    ("「MCPクライアント」とは何ですか？", ["サーバーに接続"]),
    ("「fork」とは何ですか？", ["コピー"]),
    ("LoRAについて簡単に説明してください。", ["低ランク", "アダプタ"]),
]
# paraphrase: 用語集にある語を、学習とは違う言い回しで聞く
PARAPHRASE = [
    ("LLMって何ですか？", ["予測", "文章"]),
    ("temperatureを上げるとどうなりますか？", ["多様"]),
    ("rate limitに引っかかるとどんなエラーが出ますか？", ["429"]),
    ("LoRAのrは何を決める値ですか？", ["低ランク行列", "大きさ"]),
    ("QLoRAとLoRAの違いは何ですか？", ["量子化", "4ビット"]),
    ("エポック数を増やしすぎるとどうなりますか？", ["過学習"]),
    ("SFTではどんなデータを使いますか？", ["入力", "出力"]),
    ("PEFTの利点は何ですか？", ["メモリ"]),
    ("Codespacesに登録したシークレットはどのように渡されますか？", ["環境変数"]),
    ("devcontainer.jsonには何を書きますか？", ["Python", "拡張機能"]),
    ("エージェントループはどんな処理の繰り返しですか？", ["ツール", "最終回答"]),
    ("Human-in-the-loopはどんなときに使いますか？", ["取り消しにくい"]),
    ("TestModelを使うと何ができますか？", ["APIキー"]),
    ("stdioとStreamable HTTPの違いを教えてください。", ["サブプロセス", "URL"]),
    ("ファインチューニングで最新の事実を覚えさせるのは向いていますか？", ["RAG"]),
]
# unknown: 用語集にない語（質問, 用語集に出てこないことを確かめる語）
UNKNOWN = [
    ("「Transformer」とは何ですか？", "Transformer"),
    ("「RLHF」とは何ですか？", "RLHF"),
    ("「Docker」とは何ですか？", "Docker"),
    ("「ベクトルデータベース」とは何ですか？", "ベクトル"),
    ("「LangChain」とは何ですか？", "LangChain"),
    ("プロンプトインジェクションについて簡単に説明してください。", "インジェクション"),
    ("「Kubernetes」とは何ですか？", "Kubernetes"),
    ("知識蒸留について簡単に説明してください。", "蒸留"),
]

test, gold = [], {}
for q, words in SAME:
    assert q in train_q, f"same の質問が train にありません: {q}"
    assert all(w in train_q[q] for w in words), f"語句が正解文にありません: {q}"
    test.append({"question": q, "category": "same"}); gold[q] = {"category": "same", "must_include": words}
for q, words in PARAPHRASE:
    assert q not in train_q and q not in valid_q, f"paraphrase の質問が学習データにあります: {q}"
    test.append({"question": q, "category": "paraphrase"}); gold[q] = {"category": "paraphrase", "must_include": words}
for q, word in UNKNOWN:
    assert word not in all_text, f"unknown の語が用語集に出てきます: {word}"
    test.append({"question": q, "category": "unknown"}); gold[q] = {"category": "unknown", "must_include": []}
assert len(gold) == len(test), "test の質問が重複しています"

from collections import Counter
print(f"train={len(train_rows)} valid={len(valid_rows)} test={len(test)}", dict(Counter(e["category"] for e in test)))
print("学習例の 1 件 | 質問:", q_of(train_rows[0]), "| 望ましい出力:", a_of(train_rows[0])[:60], "...")
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train_rows), encoding="utf-8")
_ = (WORK / "valid.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in valid_rows), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=110 valid=6 test=33 {'same': 10, 'paraphrase': 15, 'unknown': 8}
学習例の 1 件 | 質問: 「AIエンジニアリング」とは何ですか？ | 望ましい出力: 学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技 ...


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 33 問に答えさせて採点します。

演習と違い、プロンプトに FAQ も形式の指示も入れません。質問だけを渡します（`build_messages`）。学習時と評価時で入力の形を揃えるためです。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- `same` / `paraphrase`: 回答に `must_include` の語句がすべて含まれていれば ○。語句を含むかだけを見るので、言い換えて正しく答えても × になることがあります。表で自分の目で確かめてください。
- `unknown`: 正解が無いので、回答に「分かりません」「記載がありません」「確認」などの断り表現があれば `declined`、なければ `answered` に分類します。これは簡易な分類です。`answered` の中身（作り話か、一般知識として正しい説明か）は、表で自分の目で確かめてください。

**見るところ**

- 1 つ目のセル: モデルの読み込み（約 1.75GB のダウンロード）。
- 2 つ目のセル: 種類ごとの正解率の要約と、33 問の出力の表。回答が長いので、評価には演習より時間がかかります（数分）。

演習のノートブックと同じ警告（`HF Hub` への未認証アクセス、線形注意の実装の案内）が出ても問題ありません。

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 質問だけを渡す（system も FAQ も入れない）。学習時も同じ形で渡す
def build_messages(example):
    return [{"role": "user", "content": example["question"]}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [7]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

# unknown の分類に使う断り表現（簡易。answered の中身は表で目視する）
DECLINE_WORDS = ["わかりません", "分かりません", "知りません", "記載", "情報がありません", "確認", "存在しません", "見当たり"]

def check_content(answer: str, gold_entry) -> bool:
    '''same / paraphrase の合否: 必須語句がすべて回答に含まれるか'''
    return all(w in answer for w in gold_entry["must_include"])

def classify_unknown(answer: str) -> str:
    '''unknown の分類: 断り表現があれば declined、なければ answered'''
    return "declined" if any(w in answer for w in DECLINE_WORDS) else "answered"

def mark(ok) -> str:
    return "-" if ok is None else ("○" if ok else "×")

def evaluate(model, label: str):
    '''test の全問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        g = gold[ex["question"]]
        if g["category"] == "unknown":
            ok, kind = None, classify_unknown(raw)
        else:
            ok, kind = check_content(raw, g), ""
        rows.append({"question": ex["question"], "category": g["category"], "raw": raw, "content_ok": ok, "kind": kind})
    def rate(cat):
        sel = [r for r in rows if r["category"] == cat]
        return round(sum(r["content_ok"] for r in sel) / len(sel), 3)
    n_unknown = sum(r["category"] == "unknown" for r in rows)
    summary = {"label": label, "n": len(rows),
               "same_rate": rate("same"), "paraphrase_rate": rate("paraphrase"),
               "unknown_declined": sum(r["kind"] == "declined" for r in rows), "unknown_answered": sum(r["kind"] == "answered" for r in rows),
               "unknown_n": n_unknown, "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。same / paraphrase は必須語句の有無、unknown は断ったかどうかを、出力の原文と見比べる
display(pd.DataFrame([{"種類": r["category"], "質問": r["question"], "学習前の出力": r["raw"],
                       "内容": mark(r["content_ok"]), "unknown の分類": r["kind"]} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 33, "same_rate": 0.1, "paraphrase_rate": 0.067, "unknown_declined": 0, "unknown_answered": 8, "unknown_n": 8, "seconds": 404.4}


,種類,質問,学習前の出力,内容,unknown の分類
0,same,「トークン」とは何ですか？,"**トークン（Token）** は、インターネット上で情報やデータを送信する際に使用される**「単位」**または**「通貨」**のことです。\n\n一般的に、以下の 3 つの用途で使われます：\n\n1. **Web 通信（Web 3.0）**\n * 仮想通貨（例：Bitcoin, Ethereum）やブロックチェーン上の取引単位として使われます。\n * 例：Web3.0 の取引では、トークンが「取引単位」として機能し、取引手数料や価格変動に応じて値が変動します。\n\n2. **Web アプリケーション**\n * アプリケーションのユーザー認証やログイン時に使用される単位です。\n * 例：Google の Gmail 認証では、トークンがユーザーの認証情報として使用されます。\n\n3. **ゲームやアプリ**\n * ゲームやアプリのゲーム内通貨として使われます。\n * 例：ゲーム内では「トークン」がゲームの報酬やアイテムとして扱われることがあります。\n\n**まとめると**、トークンは「通信の単位」や「取引の単位」であり、インターネット上で情報をやり取りする際に使われる抽象的な単位です。",×,
1,same,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに**過剰に適合**している。\n2. **欠学習**：モデルが学習したデータに**欠けている**（真のデータに欠ける）。\n3. **過学習・欠学習**：両者の両方がある。\n\nこの状態は、モデルが「学習したデータ」に**過剰に適合**しているため、**真のデータに欠ける**状態を指します。\n\n### 過学習の例\n例えば、「猫は犬ではない」という事実を学習したモデル（「猫は犬ではない」というルールをすべて適用した）が、真の猫と犬のデータに欠けるという状態です。\n\n### 解決策\n過学習を防ぐために、以下の方法が一般的です。\n*,×,
2,same,「ハルシネーション」とは何ですか？,**ハルシネーション**（Hārinēshō）は、日本語で「**ハルシネーション**」です。\n\nこの言葉は、主に**物理学**（特に光の現象）や**音楽学**（特に弦楽器の音）において使われます。\n\n* **物理学の定義**: 光の波が媒質（空気、水など）を通過する際、波の振幅が媒質の密度や温度によって変化し、波の長さが変化する現象です。\n * 例：太陽光が空気中に進むと、波長が短くなり、波の振幅が小さくなります。\n * 例：水波が波紋を形成する際、波の振幅が波紋の深さ（ハルシネーション）に比例して変化します。\n* **音楽学的定義**: 弦楽器（特にピアノやギターなど）の音源において、音の振幅が音の周波数（音の高さ）に比例して変化し、音の形状が変化する現象です。\n * 例：ピアノの音源において、音の周波数が高くなるほど、音の振幅が小さ,×,
3,same,「APIキー」とは何ですか？,**API キー**（Application API Key）とは、アプリケーション（アプリやサービス）が、外部のサービス（API）にアクセスするための**認証情報**です。\n\n簡単に言うと、「あなたのアプリが誰かにデータを送るためのパスワード」や「アクセス権限の証明」のことです。\n\n### 主な特徴と仕組み\n\n1. **認証の必須条件**\n 多くの API サービス（例：Google API、Facebook API、AWS API など）は、ユーザーが直接サーバーにログインして認証を行うのではなく、**アプリ側から発行されたキー**を使ってアクセスできるように設計しています。\n * 例：Google の「Google Cloud Console」では、ユーザーが「Google」アカウントでログインし、そのアカウントから発行された「API キー」をアプリに渡すことで、アプリが Google のデータにアクセスできるようになります。\n\n2. **用途**\n * **データ取得・処理**: 特定のサービスからデータを取得し、そのデータを処理する際に必要です。\n * **データ送信**: 外部サービスにデータを送る際、そのサービスにアクセスするための鍵です。\n * **セキュリティ**: 鍵を管理するのではなく、アプリ側で管理し、サーバー,×,
4,same,「量子化」とは何ですか？,**「量子化」**とは、物理学において**「量子力学」**という理論の核心を捉える概念です。\n\n簡単に言うと、**「連続的なものではなく、特定の「量子」の単位でしか変化しない性質」**を指します。\n\nこの概念は、1900 年、ニュートンやプランクによって発見された**「プランク定数」**から始まります。プランクは、熱力学の第二法則（熱は連続ではない）と量子力学の第一法則（エネルギーは量子化される）を結びつけて考え、**「エネルギーは量子（量子）の単位でしか変化しない」**と結論づけたのです。\n\n### 具体的な例\n\nこの概念は、日常生活中的な現象でもそのまま使えませんが、いくつかの重要な例があります。\n\n1. **光の発光**\n 光（電磁波）は、波長や周波数という「連続的な」性質を持っています。しかし、光を**「量子（光子）」**として捉えると、そのエネルギーは特定の「量子」の単位でしか発生しません。\n * 例：紫外線（短波長）は「量子」の単位で発生し、赤外線,×,
5,same,「エポック」とは何ですか？,**「エポック」**（Epoch）は、英語の **Epoch** の日本語訳で、**「時代」「時代間」「時代間期」**を指します。\n\nこの言葉は主に**歴史**や**時間軸**において、特定の時期を指す言葉として使われます。その意味は以下の通りです。\n\n* **歴史的な意味**: 特定の時代間（例：古代、中世、近代、現代など）を指します。\n* **時間軸の単位**: 時間軸上で特定の期間を区別する単位として使われます。\n* **文化的・社会的な意味**: 特定の文化や社会現象が定着した時期を指すこともあります。\n\n例：\n* 「**中世**」は、古代から中世にかけての時代間です。\n* 「**近代**」は、中世から近代にかけての時代間です。\n* 「**現代**」は、近代から現代にかけての時代間です。\n\nまた、日本語では「時代間」や「時代間期」とも表記されることがあります。,×,
6,same,「ガードレール」とは何ですか？,**ガードレール**（ガードレール）は、主に**歩行者や車両の安全を確保するための安全装置**です。\n\nこの用語は、英語の「Guardrail」を日本語で訳したものです。その意味は、道路や歩道に設置された**金属製のフェンス**（またはコンクリート製のフェンス）であり、その役割は以下の通りです。\n\n* **衝突防止**: 歩行者や車両が衝突する際に、フェンスが衝突するのを防ぐため、設置されています。\n* **安全な通行**: 歩行者が歩道に踏む際に、歩行者が転倒するのを防ぐため、設置されています。\n* **視認性**: 歩行者が道路の境界線や危険な場所を認識し、安全に通行できるように設計されています。\n\n一般的に、**「歩道ガードレール」**と呼ばれることが多いですが、道路の両側（車道と歩道）にも設置されるため、**「道路ガードレール」**という表現も使われます。\n\n日本の法律（道路交通法）では、歩行者の安全を確保するための「歩道ガードレール」を「歩道安全装置」として規定しています。,×,
7,same,「MCPクライアント」とは何ですか？,"**MCP（Model Context Protocol）** は、大規模 AI モデル（特に大規模言語モデル）と、それらを動かす「MCP クライアント」の通信を標準化するための規格です。\n\nこの用語は主に **Microsoft Azure AI** の製品やサービス（Azure AI Studio や Azure AI Agent など）で使用されています。以下にその核心を簡潔に説明します。\n\n### 1. 基本定義\nMCP クライアントとは、AI モデルを「理解」し、その情報を処理する**人間や機械**のことです。\n一方、MCP は、AI モデルと人間（または機械）が通信するための**標準化された通信プロトコル**です。\n\n### 2. 主な特徴\n* **通信の標準化**: 従来のテキストベースのチャットや、複雑な API 連携を統一した通信方式を提供します。\n* **モデルの活用**: 特定の AI モデル（例：GPT-4, Claude など）を、そのモデルの文脈（Context）に合わせて、人間が直接操作できるように設計されています。\n* **エージェント機能**: 単なるチャットではなく、ユー

## 4. LoRA 学習

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 110 件に LoRA を付けて学習します。学習した adapter は `adapter/` に保存します。

- `glossary_sft.jsonl` はすでに `messages` 形式なので、変換せずそのまま `Dataset` にします（演習では `to_chat` で変換していました）。
- `target_modules` は LoRA を差し込む層です。演習と同じ設定です。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 110 件なので 1 エポックは 14 ステップで、5 エポックで約 70 ステップです。
- `eval_strategy="steps"` を追加して、10 ステップごとに valid の loss を出します。train の loss だけが下がり、valid の loss が下がらなくなったら、過学習のサインです。
- 学習後のモデルは変数 `tuned` に入ります。学習前の出力が必要なときは、`tuned.disable_adapter()` で adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであること。
- `Training Loss` と `Validation Loss` の表（10 ステップごと）: どちらも下がっていくか。
- `学習時間: ... 秒 | ステップ数: ... | エポック: 5.0`: 学習時間は結果に記録しておきます。

**うまくいかないとき**: 学習が止まった、または学習できなかった場合は、下の 4b を読んでください。

In [8]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 学習データはすでに messages 形式（user / assistant）なので、そのまま Dataset にする
train_ds = Dataset.from_list(train_rows)
valid_ds = Dataset.from_list(valid_rows)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, eval_strategy="steps", eval_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,3.232859,2.618922,2.889850,6023.000000,0.512195
20,2.237584,1.986943,2.059684,11935.000000,0.614191
30,1.606717,1.573056,1.831023,17890.000000,0.700665
40,0.967486,1.158985,1.472690,23858.000000,0.782705
50,0.670777,1.081243,1.338069,29743.000000,0.787140
60,0.473168,0.889001,1.257058,35546.000000,0.829268
70,0.400068,0.870691,1.223057,41555.000000,0.842572


学習時間: 174 秒 | ステップ数: 70 | エポック: 5.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 設定したエポック数まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 33 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `same_rate`（学習した質問の再現）、`paraphrase_rate`（言い換えへの対応）が学習前からどう変わったか。`unknown_declined` / `unknown_answered` は、用語集にない語を断った数と、答えた数です。
- 質問ごとの表: `same` と `paraphrase` の失敗は、必須語句が欠けたのか、言い換えて正しく答えているのに語句が合わないだけなのかを、出力の原文で読み分けてください。
- `paraphrase_rate` が `same_rate` より低ければ、学習した文面の暗記にとどまっている可能性があります。

seed を固定しても、GPU 上の学習は完全には再現しません。割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。

In [9]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")
display(compare)

# 質問ごとの出力原文
detail = pd.DataFrame([{"種類": b["category"], "質問": b["question"],
                        "学習前の出力": b["raw"], "学習前": mark(b["content_ok"]) if b["category"] != "unknown" else b["kind"],
                        "学習後の出力": a["raw"], "学習後": mark(a["content_ok"]) if a["category"] != "unknown" else a["kind"]}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 33, "same_rate": 0.7, "paraphrase_rate": 0.2, "unknown_declined": 0, "unknown_answered": 8, "unknown_n": 8, "seconds": 125.3}


,n,same_rate,paraphrase_rate,unknown_declined,unknown_answered,unknown_n,seconds
label,,,,,,,
before,33,0.1,0.067,0,8,8,404.4
after,33,0.7,0.200,0,8,8,125.3


,種類,質問,学習前の出力,学習前,学習後の出力,学習後
0,same,「トークン」とは何ですか？,"**トークン（Token）** は、インターネット上で情報やデータを送信する際に使用される**「単位」**または**「通貨」**のことです。\n\n一般的に、以下の 3 つの用途で使われます：\n\n1. **Web 通信（Web 3.0）**\n * 仮想通貨（例：Bitcoin, Ethereum）やブロックチェーン上の取引単位として使われます。\n * 例：Web3.0 の取引では、トークンが「取引単位」として機能し、取引手数料や価格変動に応じて値が変動します。\n\n2. **Web アプリケーション**\n * アプリケーションのユーザー認証やログイン時に使用される単位です。\n * 例：Google の Gmail 認証では、トークンがユーザーの認証情報として使用されます。\n\n3. **ゲームやアプリ**\n * ゲームやアプリのゲーム内通貨として使われます。\n * 例：ゲーム内では「トークン」がゲームの報酬やアイテムとして扱われることがあります。\n\n**まとめると**、トークンは「通信の単位」や「取引の単位」であり、インターネット上で情報をやり取りする際に使われる抽象的な単位です。",×,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、モデルの特性（トークン数）に関わる。,○
1,same,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに**過剰に適合**している。\n2. **欠学習**：モデルが学習したデータに**欠けている**（真のデータに欠ける）。\n3. **過学習・欠学習**：両者の両方がある。\n\nこの状態は、モデルが「学習したデータ」に**過剰に適合**しているため、**真のデータに欠ける**状態を指します。\n\n### 過学習の例\n例えば、「猫は犬ではない」という事実を学習したモデル（「猫は犬ではない」というルールをすべて適用した）が、真の猫と犬のデータに欠けるという状態です。\n\n### 解決策\n過学習を防ぐために、以下の方法が一般的です。\n*,×,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,○
2,same,「ハルシネーション」とは何ですか？,**ハルシネーション**（Hārinēshō）は、日本語で「**ハルシネーション**」です。\n\nこの言葉は、主に**物理学**（特に光の現象）や**音楽学**（特に弦楽器の音）において使われます。\n\n* **物理学の定義**: 光の波が媒質（空気、水など）を通過する際、波の振幅が媒質の密度や温度によって変化し、波の長さが変化する現象です。\n * 例：太陽光が空気中に進むと、波長が短くなり、波の振幅が小さくなります。\n * 例：水波が波紋を形成する際、波の振幅が波紋の深さ（ハルシネーション）に比例して変化します。\n* **音楽学的定義**: 弦楽器（特にピアノやギターなど）の音源において、音の振幅が音の周波数（音の高さ）に比例して変化し、音の形状が変化する現象です。\n * 例：ピアノの音源において、音の周波数が高くなるほど、音の振幅が小さ,×,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や部門に固有の情報など）を、自分の口調で広めるために起こる。本講座では、ツールを呼び出す前にモデルに明確に伝えることで、ハルシネーションを減らす。,○
3,same,「APIキー」とは何ですか？,**API キー**（Application API Key）とは、アプリケーション（アプリやサービス）が、外部のサービス（API）にアクセスするための**認証情報**です。\n\n簡単に言うと、「あなたのアプリが誰かにデータを送るためのパスワード」や「アクセス権限の証明」のことです。\n\n### 主な特徴と仕組み\n\n1. **認証の必須条件**\n 多くの API サービス（例：Google API、Facebook API、AWS API など）は、ユーザーが直接サーバーにログインして認証を行うのではなく、**アプリ側から発行されたキー**を使ってアクセスできるように設計しています。\n * 例：Google の「Google Cloud Console」では、ユーザーが「Google」アカウントでログインし、そのアカウントから発行された「API キー」をアプリに渡すことで、アプリが Google のデータにアクセスできるようになります。\n\n2. **用途**\n * **データ取得・処理**: 特定のサービスからデータを取得し、そのデータを処理する際に必要です。\n * **データ送信**: 外部サービスにデータを送る際、そのサービスにアクセスするための鍵です。\n * **セキュリティ**: 鍵を管理するのではなく、アプリ側で管理し、サーバー,×,APIを呼び出す先を明示的に指定する秘密の文字列。漏れると他人がツールを呼び出し、その結果を読むなどの攻撃を始める。本講座では「設定ファイルの secret 分野」に書く。,○
4,same,「量子化」とは何ですか？,**「量子化」**とは、物理学において**「量子力学」**という理論の核心を捉える概念です。\n\n簡単に言うと、**「連続的なものではなく、特定の「量子」の単位でしか変化しない性質」**を指します。\n\nこの概念は、1900 年、ニュートンやプランクによって発見された**「プランク定数」**から始まります。プランクは、熱力学の第二法則（熱は連続ではない）と量子力学の第一法則（エネルギーは量子化される）を結びつけて考え、**「エネルギーは量子（量子）の単位でしか変化しない」**と結論づけたのです。\n\n### 具体的な例\n\nこの概念は、日常生活中的な現象でもそのまま使えませんが、いくつかの重要な例があります。\n\n1. **光の発光**\n 光（電磁波）は、波長や周波数という「連続的な」性質を持っています。しかし、光を**「量子（光子）」**として捉えると、そのエネルギーは特定の「量子」の単位でしか発生しません。\n * 例：紫外線（短波長）は「量子」の単位で発生し、赤外線,×,モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。,○
5,same,「エポック」とは何ですか？,**「エポック」**（Epoch）は、英語の **Epoch** の日本語訳で、**「時代」「時代間」「時代間期」**を指します。\n\nこの言葉は主に**歴史**や**時間軸**において、特定の時期を指す言葉として使われます。その意味は以下の通りです。\n\n* **歴史的な意味**: 特定の時代間（例：古代、中世、近代、現代など）を指します。\n* **時間軸の単位**: 時間軸上で特定の期間を区別する単位として使われます。\n* **文化的・社会的な意味**: 特定の文化や社会現象が定着した時期を指すこともあります。\n\n例：\n* 「**中世**」は、古代から中世にかけての時代間です。\n* 「**近代**」は、中世から近代にかけての時代間です。\n* 「**現代**」は、近代から現代にかけての時代間です。\n\nまた、日本語では「時代間」や「時代間期」とも表記されることがあります。,×,学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。,○
6,same,「ガードレール」とは何ですか？,**ガードレール**（ガードレール）は、主に**歩行者や車両の安全を確保するための安全装置**です。\n\nこの用語は、英語の「Guardrail」を日本語で訳したものです。その意味は、道路や歩道に設置された**金属製のフェンス**（またはコンクリート製のフェンス）であり、その役割は以下の通りです。\n\n* **衝突防止**: 歩行者や車両が衝突する際に、フェンスが衝突するのを防ぐため、設置されています。\n* **安全な通行**: 歩行者が歩道に踏む際に、歩行者が転倒するのを防ぐため、設置されています。\n* **視認性**: 歩行者が道路の境界線や危険な場所を認識し、安全に通行できるように設計されています。\n\n一般的に、**「歩道ガードレール」**と呼ばれることが多いですが、道路の両側（車道と歩道）にも設置されるため、**「道路ガードレール」**という表現も使われます。\n\n日本の法律（道路交通法）では、歩行者の安全を確保するための「歩道ガードレール」を「歩道安全装置」として規定しています。,×,エージェントの行動を一定の範囲以内に抑える仕組み。指示文で定める（制約文、出力制限、外部データへのアクセス制限など）。本講座では「ツールの使用量・出力の長さ・外部データへのアクセス」をモデル

## 6. 用語集にない語（学習データに無い知識は保証されない）

**やること**: `unknown` の 8 問だけを取り出し、学習前後の出力を並べます。学習データには「分かりません」と答える例を入れていないので、学習後に断る形になるとは限りません。

**見るところ**

- `answered` になった質問は、何を根拠に答えたように見えるか。用語集の近い語の定義を流用していないか、もっともらしい作り話になっていないか。
- 学習前は一般知識として正しく答えられていた語が、学習後に用語集の文体で崩れていないか。

「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。学習前の出力は、セクション 3 で取った `before_rows` を使います。

In [10]:
unknown_rows = [{"質問": b["question"], "学習前の分類": b["kind"], "学習前の出力": b["raw"],
                 "学習後の分類": a["kind"], "学習後の出力": a["raw"]}
                for b, a in zip(before_rows, after_rows) if b["category"] == "unknown"]
display(pd.DataFrame(unknown_rows))

,質問,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,「Transformer」とは何ですか？,answered,"**Transformer** は、**大規模言語モデル（LLM）**の核心となる技術の一つです。\n\n簡単に言うと、**「文脈を理解し、自然な日本語で説明する能力」**を備えた新しい学習モデルです。従来の機械学習モデル（例えば、ニューラルネットワークやディープラーニング）では、文脈を考慮して情報を処理するのではなく、単に「入力 → 出力」の直線的な計算を行っていました。\n\nでは、なぜ Transformer が注目されているのか、その理由と仕組みを解説します。\n\n### 1. 従来のモデルとの違い\n* **従来のモデル（例：BERT, RoBERTa）**: 文脈を考慮して単語の位置や順序を分析するモデルです。\n* **Transformer**: 文脈を考慮せず、**「単語の順序」を重視**するモデルです。\n\n### 2. 仕組み：Attention Mechanism\nTransformer の最大の強みは、**「Attention（注目）」**というメカニズムです。\n* 入力文脈をすべて考慮するのではなく、**「どの単語が文脈の中で最も重要か」**を判断する仕組みです。\n* 入力文脈の単語をすべて",answered,LLMの処理を1つの層にまとめるモデル。2017年に提出され、AIE2017で世界大会に参戦。2022年よりLoRAで重量を減らす軽量版LoRA-Transformerが公開される。
1,「RLHF」とは何ですか？,answered,**RLHF** は、**Reinforcement Learning from Human Feedback（人間からのフィードバックから得られるリフレクション学習）** の略です。\n\nこれは、大規模言語モデル（LLM）などの AI 機器が人間がどのように評価し、どのように指示するかの「人間との対話」をデータとして学習し、その学習されたモデルを、人間が指示した方向に調整する技術です。\n\n具体例として、以下の 3 つのステップで説明します。\n\n1. **人間との対話**：\n 人間が AI に質問したり、指示したり、あるいは特定の文脈で話したりします。この対話の出力（回答や指示）を記録します。\n\n2. **学習**：\n AI が、記録された対話データを使って、学習アルゴリズム（例えば DPO、PPO など）を使って、人間が指示した方向に調整します。\n\n3. **調整**：\n 学習されたモデルが、人間が指示した方向に調整された状態に戻ります。\n\n**主な目的**\n* **人間との対話の理解**：AI が人間が何を言いたいのか、どのように指示すべきかを理解できます。,answered,モデルの出力を人の意見や評価で改善するループ。指示の書き方、評価の仕方、人の参加の質が大きく変わる。
2,「Docker」とは何ですか？,answered,**Docker** は、**「Dockerfile」の略称**です。\n\nこれは、Linux 環境において、**コンテナ（Container）**を構築するためのツールです。コンテナとは、OS の環境を完全に切り替えることなく、特定のアプリケーションやサービスを実行するための小さな、独立した「容器」のことです。\n\nDocker は、以下の 3 つの主要な機能を備えています：\n\n1. **コンテナの構築**：\n Dockerfile を利用して、アプリケーションを構築し、そのコンテナを生成します。\n ```bash\n docker build -t myapp:latest .\n ```\n\n2. **コンテナの起動**：\n 構築されたコンテナを起動し、その環境を起動します。\n ```bash\n docker run -d --name myapp myapp:latest\n ```\n\n3. **コンテナの管理**：\n コンテナの状態を監視し、起動/停止、リロードなどの操作を行います。\n ```bash\n docker ps\n docker logs myapp:latest\n docker restart myapp:latest\n ```\n\n**主な利点**\n* **コンテナの隔離**：,answered,仮想環境の定義と起動をコードに任せ、他社の镜像を問わずすべての環境が同じになるためのプラットフォーム。
3,「ベクトルデータベース」とは何ですか？,answered,**ベクトルデータベース**（Vector Database）は、データが「ベクトル（数値列）」の形式で保存・検索されるデータベースシステムです。\n\n従来の「テーブル（行）ベース」のデータベース（例：SQL データベース）とは異なり、データが空間的な位置や方向性を表現するベクトルとして扱われます。これにより、データが物理的な空間（座標）に位置づけることができます。\n\n主な特徴と用途は以下の通りです。\n\n### 主な特徴\n1. **空間的表現**: 数値列をベクトルとして扱うことで、データが「どこにあるか」や「どこから来たか」を表現できます。\n2. **検索効率**: 検索クエリ（例：「このデータが 2024 年のデータで、東京に位置するもの」）を高速に処理できます。\n3. **画像・音声の対応**: 画像や音声データもベクトルとして保存・検索可能（特に Deep Learning 技術を活用した場合）。\n4. **ノイズの耐え**: 物理的な空間（座標）を基準にすることで、データ内のノイズや誤差を,answered,大量のテキストや画像を共通のフォーマットで扱えるデータベース。AIがデータを読み込むとき、どの形式かを決めるかによって性能とコストが大きく変わる。Pydantic AIでは model_column 指定の型に依存する。
4,「LangChain」とは何ですか？,answered,**LangChain** は、**Python** を使用して、**大規模な言語モデル（LLM）のコードを生成・実行する**ための**汎用的なフレームワーク**です。\n\n簡単に言うと、LLM を単なるテキスト処理ツールではなく、**「コードの生成・実行」に使えるツール**として設計されています。\n\n主な特徴と用途は以下の通りです。\n\n### 1. 核心機能：コード生成と実行\nLangChain は、LLM（大規模言語モデル）の出力をそのまま実行するのではなく、**「コードを生成する」**という機能を強化しています。\n* **コード生成**: LLM が特定のタスク（例：「ユーザーのパスワードを生成する」）に対して、コードを自動で生成します。\n* **コード実行**: 生成されたコードを実行し、結果を出力します。\n* **タスク実行**: 複数のタスクを個別に実行し、結果をまとめて出力します。\n\n### 2. 主な用途\n* **AI アプリの作成**: 特定の業務プロセス（例：顧客の問い合わせ対応、データ分析、自動化）を実現するアプリケーションを構築します。\n* **自動化**: 人間が手,answered,MCPのツールをLLMに渡すプラットフォーム。MCPサーバーをLangChainのプログラムに貼ることで、ツール呼び出しをコードのように書ibleにし、エージェントのループやプロンプトで制御できる。
5,プロンプトインジェクションについて簡単に説明してください。,answered,**プロンプトインジェクション**（Prompt Injection）とは、ユーザーが入力する「プロンプト（指示）」を、攻撃者が制御して利用する技術です。\n\n簡単に言うと、**「あなたが特定の目的のために、私の入力内容をそのままコピーして使う」という意図を込めて、攻撃者があなたの入力内容をそのままコピーして使う」**というものです。\n\n### 仕組みと原理\n攻撃者は、攻撃者の入力内容（例：パスワード、パスワードの生成コード、特定のコード）を、攻撃者の入力内容として利用するよう、プロンプトインジェクションに設定します。\n\n* **攻撃者の入力**: 「パスワードは、攻撃者の入力内容です。」\n* **攻撃者の利用**: 「パスワードは、攻撃者の入力内容です。」\n\nこのように、攻撃者が「あなたの入力内容」をそのままコピーして使うよう指示を出し、攻撃者がそのコピーをコピーして利用します。\n\n### 目的\n* **パスワードの生成**: 攻撃者がパスワードを生成する際に、攻撃者の入力内容を使用する。\n* **コードの生成**: 攻撃者がコードを生成する際に、攻撃者の入力内容を使用する。\n* **データ窃取**: 攻撃者が特定のデータ（例,answered,LLMがツールを呼び出す時の入力文を、利用者の口調や目的に合わせて変える技術。指示文の選び方、ツール名の付け方、呼び出し時のメンター（追加文）の使い分けが、プロンプトインジェクションの成果に大きく影響する。
6,「Kubernetes」とは何ですか？,answered,**Kubernetes** は、**「Kubernetes」**という名前の由来から、**「Kubernetes」

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、データの件数を 1 つの `result.json` にまとめます。

**まとめるもの**

- `result.json`、`eval_before.json`、`eval_after.json`、`test.jsonl`、`gold.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード）
- セクション 5 の比較表と、`same` / `paraphrase` の失敗の説明
- セクション 6 の結果（断った応答、作り話が残った応答）

In [11]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"train": len(train_rows), "valid": len(valid_rows), "test": dict(Counter(e["category"] for e in test))},
    "before": before_summary, "after": after_summary,
    "unknown": unknown_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "unknown"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    },
    "glossary_sft_path": "/content/glossary_sft.jsonl"
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "data": {
    "train": 110,
    "valid": 6,
    "test": {
      "same": 10,
      "paraphrase": 15,
      "unknown": 8
    }
  },
  "before": {
    "label": "before",
    "n": 33,
    "same_rate": 0.1,
    "paraphrase_rate": 0.067,
    "unknown_declined": 0,
    "unknown_answered": 8,
    "unknown_n": 8,
    "seconds": 404.4
  },
  "after": {
    "label": "after",
    "n": 33,
    "same_rate": 0.7,
    "paraphrase_rate": 0.2,
    "unknown_declined": 